<a href="https://colab.research.google.com/github/nadiduno/TCCFlyGrupo4/blob/main/tcc_evasaofly_g4cienciadedadosfly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Ada Lovelace - Turma Fly · diversiData

> 🗺️ **Autoras:** [Brenda Amaral](https://www.linkedin.com/in/brendaamarals/), [Fernanda da Silva](https://www.linkedin.com/in/fernanda-leticia-silva/), [Nadiveth Duno](https://www.linkedin.com/in/nadiduno/), Profana Buzato, [Sheilliane Santos](https://www.linkedin.com/in/sheillianesantos/), [Vicencia Vitória Souza](www.linkedin.com/in/vicencia-vitoria).

> 🗺️**Orientadora:** [Andressa Freires](https://www.linkedin.com/in/andressafreires/)


Um modelo preditivo de potencial de empregabilidade e mobilidade financeira para egressos da Fly Educação

---
# 📥 1. Carregar os dados


In [ ]:
!pip install missingno gdown pyarrow openpyxl -q
print("Pacotes instalados!")


Pacotes instalados!


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from google.colab import files
import warnings
import urllib.parse
import sys
import os

warnings.filterwarnings('ignore')

print(f"pandas {pd.__version__}, numpy {np.__version__}, seaborn {sns.__version__}")
print('Importação com sucesso!')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')
print(f'   seaborn {sns.__version__}')

pandas 2.2.2, numpy 2.0.2, seaborn 0.13.2
Importação com sucesso!
   pandas  2.2.2
   numpy   2.0.2
   seaborn 0.13.2


In [ ]:
sheet_id = "1y0cfHPKjqp75TlMEzBdOMVuifPB_P__P"
abas = {"T10_11_14_15_17": "P1_turmas_10_11_14_15_17","T12": "P2_turma_12","T16": "P3_turma_16","T18": "P4_turma_18","T19": "P5_turma_19","T20": "P6_turma_20","T22": "P7_turma_22","T23": "P9_turma_23",}
dataFlyTurmas = {}

for aba, planilha_id in abas.items():
    nome_codificado = urllib.parse.quote(aba)
    url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={nome_codificado}"
    try:
        df = pd.read_csv(url)
        df["turma_aba"] = aba
        dataFlyTurmas[aba] = df
        #print(f"{aba}-> {df.shape[0]} linhas, {df.shape[1]} colunas")
    except Exception as e:
        print(f"{aba}: ERRO -> {e}")
print("Dados carregados com sucesso")

Dados carregados com sucesso


In [ ]:
#dataFlyTurmas["T10_11_14_15_17"].head(3)

## Usar script para renomear colunas

---



In [ ]:
# Baixar Script de Pythom com novos nomes para as colunas  renameCols_maps.py (id 10VCq_huDlCFg7y9QGbLosQnwAU0fmzoe)
!gdown 10VCq_huDlCFg7y9QGbLosQnwAU0fmzoe
from renameCols_maps import rename_maps
!ls -la *.py
print("Script executado")

Downloading...
From (original): https://drive.google.com/uc?id=10VCq_huDlCFg7y9QGbLosQnwAU0fmzoe
From (redirected): https://drive.google.com/uc?id=10VCq_huDlCFg7y9QGbLosQnwAU0fmzoe&confirm=t&uuid=60119249-9e8d-4cfb-ac9c-4d1125b8612c
To: /content/renameCols_maps.py
100% 38.5k/38.5k [00:00<00:00, 116MB/s]
-rw-r--r-- 1 root root 38535 Aug  9 04:38 renameCols_maps.py
Script executado


In [ ]:
# Renomeando as colunas usando o Script
dataFlyRenamed = {}

for aba, df in dataFlyTurmas.items():
    df_r = df.rename(columns=rename_maps[aba])
    dataFlyRenamed[aba] = df_r
    # checagem: coluna renomeada
    nao_renomeadas = [c for c in df_r.columns if c in rename_maps[aba].values()][:0]  # placeholder
    cols_originais_restantes = set(df.columns) & set(df_r.columns)  # nomes que sobreviveram sem mudar
print("As colunas foram renomeadas conforme o novo mapeamento")

As colunas foram renomeadas conforme o novo mapeamento


In [ ]:
#Juntar todas abas em um novo DataFRame
if dataFlyRenamed:
    dataFlyJoin = pd.concat(
        dataFlyRenamed.values(),
        ignore_index=True,
        sort=False
    )
else:
    dataFlyJoin = pd.DataFrame() # Initialize as an empty DataFrame if no data to concatenate
    print("Advertencia: dataFlyRenamed está vacío. dataFlyJoin se ha inicializado como un DataFrame vacío.")


print(f"Abas unificadas: {dataFlyJoin.shape[0]} linhas, {dataFlyJoin.shape[1]} colunas")

Abas unificadas: 2348 linhas, 91 colunas


In [ ]:
dataFlyJoin.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2348 entries, 0 to 2347
Data columns (total 91 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   nome_completo                            2348 non-null   object 
 1   status_aprovacao                         1170 non-null   object 
 2   genero                                   2229 non-null   object 
 3   lgbtqia                                  2228 non-null   object 
 4   raca_etnia                               2347 non-null   object 
 5   idade                                    1976 non-null   object 
 6   pcd                                      2222 non-null   object 
 7   pcd_tipo                                 862 non-null    object 
 8   pcd_impacto_aprendizagem                 211 non-null    object 
 9   escolaridade                             2347 non-null   object 
 10  estado_residencia                        2162 no

## Salvar e usar Parquet

In [ ]:
dataFlyJoin['idade'] = pd.to_numeric(dataFlyJoin['idade'], errors='coerce').astype('Int64')
dataFlyJoin['qtd_pessoas_casa'] = pd.to_numeric(dataFlyJoin['qtd_pessoas_casa'], errors='coerce').round().astype('Int64')
dataFlyJoin['data_nascimento'] = pd.to_datetime(dataFlyJoin['data_nascimento'], errors='coerce')

dataFlyJoin.to_csv('dataFlyRaw.csv', index=False, encoding='utf-8-sig')
dataFlyJoin.to_parquet('dataFlyRaw.parquet', index=False)
print("Arquivo CSV e Parquet salvos com sucesso - Data Raw!")

Arquivo CSV e Parquet salvos com sucesso - Data Raw!


In [ ]:
dataFlyRaw = pd.read_parquet('dataFlyRaw.parquet')
dataFlyRaw.to_parquet('dataFlyRaw.parquet', index=False, compression='gzip')

In [ ]:
dataFlyRaw.to_csv('dataFlyRaw.csv', index=False, encoding='utf-8-sig')
files.download('dataFlyRaw.csv')
files.download('dataFlyRaw.parquet')
print("Descarga concluida!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descarga concluida!


---
# 🔍 2. Conhecer os dados (o raio-x)

In [ ]:
# ── 2. CONHECER ─────────────────────────────────────────────────────────────
# df.shape           # (linhas, colunas) → tem o tamanho que você esperava?
# df.head(3)         # a "cara" dos dados
# df.info()          # tipos das colunas → número está como número? (não como 'object')
# df.describe()      # resumo dos números → tem mínimo/máximo impossível? (outliers)
# df.isnull().sum()  # quantos VAZIOS por coluna → onde precisa tratar?
# df.nunique()       # quantos valores DIFERENTES por coluna → alguma quase constante?
# df['COLUNA'].value_counts()  # o que tem numa categoria → texto repetido escrito diferente?

In [ ]:
# Seu aporte aqui

---
# 🧹 3. Limpar os dados

> **Objetivo:** corrigir os problemas que você achou no passo 2.
> **Consulte:** notebook da aula de limpeza para exemplos completos.

In [ ]:
# ── 3. LIMPAR ───────────────────────────────────────────────────────────────
# Duplicatas:
# df = df.drop_duplicates()

# Tipos (texto → número; extrair dígitos de textos como '28 anos'):
# df['COLUNA'] = df['COLUNA'].astype(str).str.extract(r'(\d+)').astype('Int64')

# Nulos — escolha a estratégia por coluna:
# df['CATEGORICA'] = df['CATEGORICA'].fillna('Não informado')        # categórico: rótulo com sentido
# df['NUMERICA']   = df['NUMERICA'].fillna(df['NUMERICA'].median())  # número: mediana (robusta)
# df = df.dropna(subset=['COLUNA_ESSENCIAL'])                        # remove sem info essencial

# Padronizar texto (evita 'Belém' ≠ 'belem' na hora de juntar):
# df['COLUNA'] = df['COLUNA'].str.strip().str.lower()               # tira espaços + minúsculo
# (para remover acentos, veja a função da aula de limpeza)

# 👉 Confira ao final:  df.isnull().sum().sum()   # ideal: 0

In [ ]:
# Seu aporte aqui

---
# 🧩 4. Combinar fontes (montar a base de análise)

> **Objetivo:** deixar a base na granularidade certa — **um caso por linha** (ex.: um aluno por linha).
> **Pergunta-chave:** "o que é UMA linha aqui?" vs "o que precisa ser uma linha pra minha análise?"
> **Consulte:** Guia HTML (seção "Combinar") para o passo a passo com exemplos.


In [ ]:
# ── 4. COMBINAR ─────────────────────────────────────────────────────────────

# 4a) EMPILHAR arquivos com as MESMAS colunas (mais casos):
# base = pd.concat([df_2023, df_2024], ignore_index=True)

# 4b) AGRUPAR: mudar a granularidade (de muitas linhas por caso → 1 linha por caso).
#     Padrão:  nome_novo = ('coluna_original', 'conta')
# base_analise = df.groupby('CHAVE').agg(
#     coluna_fixa = ('COLUNA',   'first'),   # info que não muda por caso
#     total       = ('VALOR',    'sum'),     # soma
#     quantidade  = ('VALOR',    'count'),   # nº de linhas do grupo
#     media       = ('NUMERICA', 'mean'),    # média
#     minimo      = ('NUMERICA', 'min'),     # menor (ex.: pior frequência)
# ).reset_index()
# Contas úteis: sum, mean, median, count, nunique, min, max, first, last, std

# 4c) JUNTAR outra tabela pela CHAVE em comum (trazer mais colunas):
#     A CHAVE é a coluna que existe nas DUAS tabelas (ex.: 'municipio', um id, o CPF).
#     Ela precisa estar escrita igual e ser do mesmo tipo nos dois lados (padronize antes!).
# base_analise = base_analise.merge(TABELA_APOIO, on='CHAVE', how='left')
#   Tipos de join (how):
#     'left'  → mantém TODOS da tabela principal (o mais comum)
#     'inner' → só quem existe nas duas (descarta em silêncio — cuidado!)
#     'outer' → todos dos dois lados | 'right' → todos da direita
#   ⚠️ confira o nº de linhas ANTES e DEPOIS do merge (df.shape)!

# 4d) REMODELAR (visão cruzada, ótima pra virar heatmap):
# visao = df.pivot_table(index='LINHAS', columns='COLUNAS', values='VALOR', aggfunc='mean')

# 🔑 Ordem de ouro: empilhar → AGRUPAR (deixa a chave única) → juntar.

---
# 📈 5. Explorar (EDA)

> **Objetivo:** descobrir o que os dados dizem — resumos, comparações entre grupos e correlações.
> **Consulte:** notebooks das aulas de EDA e estatística.


In [ ]:
# ── 5. EXPLORAR (EDA) ───────────────────────────────────────────────────────

# Tendência central e dispersão (troque 'COLUNA' pela sua variável numérica):
# base_analise['COLUNA'].mean()      # média (sensível a outliers)
# base_analise['COLUNA'].median()    # mediana (mais honesta p/ renda, salário...)
# base_analise['COLUNA'].std()       # desvio-padrão (o quanto varia)

# Comparar um número entre categorias (o coração da EDA social):
# base_analise.groupby('CATEGORIA')['COLUNA'].median().sort_values()

# Correlação entre variáveis numéricas (-1 a +1):
# base_analise[['NUM1','NUM2','NUM3']].corr()
# ⚠️ Correlação NÃO é causa — nas conclusões, escreva "sugere", nunca "prova".

---
# 📊 6. Visualizar (fechar a EDA)

> **Objetivo:** ver e contar a história. Escolha o gráfico certo pra cada pergunta.
> **Consulte:** Guia HTML (miniaturas de cada gráfico) e a aula de visualização.

In [ ]:
# ── 6. VISUALIZAR ───────────────────────────────────────────────────────────

# Histograma — distribuição de UMA variável:
# plt.hist(base_analise['COLUNA'], bins=30); plt.show()

# Boxplot — comparar grupos e ver outliers:
# sns.boxplot(data=base_analise, x='NUMERICA', y='CATEGORIA'); plt.show()

# Barras — comparar categorias:
# base_analise.groupby('CATEGORIA')['NUMERICA'].median().plot(kind='barh'); plt.show()

# Dispersão — duas numéricas se relacionam?
# plt.scatter(base_analise['NUM1'], base_analise['NUM2']); plt.show()

# Heatmap — várias correlações de uma vez:
# sns.heatmap(base_analise[['NUM1','NUM2','NUM3']].corr(), annot=True, cmap='RdBu_r', center=0); plt.show()

# 📖 Storytelling: título = conclusão · cor só no destaque · anote valores · uma ideia · cite a fonte.

---
## ✅ Checklist antes de entregar

- [ ] **Carreguei** e olhei o `head()`
- [ ] **Conheci** a base (`info`, `describe`, `isnull`, `value_counts`)
- [ ] **Limpei** (nulos, tipos, duplicatas, texto padronizado)
- [ ] **Combinei** na granularidade certa (um caso por linha) e conferi o nº de linhas nos merges
- [ ] **Explorei** (médias/medianas, comparação entre grupos, correlação)
- [ ] **Visualizei** com storytelling (título conclusivo, fonte)
- [ ] Código formatado e indentado
- [ ] Variáveis em camelCase
- [ ] URLs, Strings, JSON com aspas duplas
- [ ] Comentários claros em português
- [ ] Teste de execução da célula
- [ ] Sem dados sensíveis no código
- [ ] Changelog atualizado


**Orientadora:** [Andressa Freires - diversiData](https://www.linkedin.com/in/andressafreires/)